# Mem0 Python Memory with Oracle AI Database

This notebook shows how to build a Python memory workflow with Mem0 and Oracle AI Database.

It focuses on the Python integration path for the `mem0ai` package. The runnable notebook uses the Mem0 Python SDK, `python-oracledb`, and Oracle AI Vector Search so the same flow can move from a local notebook into an application service.

**What this notebook demonstrates**

- Connect from Python to FreeSQL, Autonomous AI Database, or a local Oracle Database.
- Configure Mem0 with the Oracle AI Database vector-store provider.
- Add memories for `User A` and `User B` with separate scopes.
- Search memories by user scope and metadata filters.
- Update and delete memories, then verify the lifecycle in Oracle.
- Inspect the Oracle table that stores the vectors and JSON payloads.

The notebook uses `infer=False` so the tutorial stores explicit memories directly. That keeps the demo easy to run in a local environment without making an LLM extraction call.

## Architecture at a Glance

```text
                 +-------------------------------+
                 | Python application / notebook |
                 | mem0ai + python-oracledb      |
                 +---------------+---------------+
                                 |
                                 | adds, searches, updates memories
                                 v
                 +-------------------------------+
                 | Mem0 Python SDK               |
                 | Memory API + embedding hook   |
                 +---------------+---------------+
                                 |
                                 | persists vectors and payload JSON
                                 v
                 +-------------------------------+
                 | Oracle AI Database            |
                 | Oracle AI Vector Search       |
                 +-------------------------------+
```

The Python application uses Mem0 as the memory layer. Mem0 calls the embedder to create vectors, then stores vectors and memory payloads in Oracle AI Database.

## What the Mem0 Python Package Provides

The Mem0 Python SDK lets Python applications add, search, update, and manage long-term memories. With the Oracle vector-store provider, those memories can be stored and searched in Oracle AI Database through Oracle AI Vector Search.

This tutorial focuses on the pieces that are most useful for a first technical notebook:

- `Memory.from_config(...)` to create a Mem0 client from Python configuration.
- `memory.add(..., infer=False)` to store explicit memories without an LLM extraction step.
- `memory.search(...)` with `filters={"user_id": ...}` to keep memory retrieval scoped.
- Metadata filters such as `category="movie"` for application-level retrieval.
- `memory.update(...)` and `memory.delete(...)` for the memory lifecycle.
- Oracle table inspection to prove the memories are persisted in the database.

## Database Connection Guide

Use one of these connection paths. The notebook reads the same `.env` variable names for all three options.

Option A - Local Oracle Database

Use a local Oracle Database when you want a repeatable development environment on your machine or in a container.

```env
ORACLE_USER=<local-user>
ORACLE_PASSWORD=<local-password>
ORACLE_DSN=localhost:1521/<service-name>
MEM0_PY_COLLECTION=MEM0_PY_MEMORY_DEMO
MEM0_PY_CREATE_INDEX=false
```

Option B - Oracle Autonomous AI Database

Use Autonomous AI Database when you want the same LangChain.js workflow against a managed Oracle AI Database environment.


Option C - FreeSQL
Use FreeSQL when you want a hosted Oracle Database schema for a tutorial without setting up a local database.

1. Sign in to FreeSQL.
2. Open Connect to the Database.
3. Choose Node.js for this notebook.
4. Copy the generated username, password, and connect descriptor.
5. Add the values to a private .env file in the same folder as this notebook.

```env
DB_USER=<freesql-user>
DB_PASSWORD=<freesql-password>
DB_CONNECT_STRING=<freesql-connect-descriptor>
MEM0_PY_COLLECTION=MEM0_PY_MEMORY_DEMO
MEM0_PY_CREATE_INDEX=false
```


## 1. Prepare the Python Runtime

The cell checks that the notebook is running from the local notebook folder, points to the parent `.env` file, and disables Mem0 telemetry before importing `mem0`. It also sets small display helpers that later cells reuse.

This step proves that the notebook can find its local configuration without hard-coding paths or credentials.

In [ ]:
from pathlib import Path
import json
import math
import os
import textwrap
import hashlib
from pprint import pprint
import logging

NOTEBOOK_DIR = Path.cwd()
ENV_PATH = NOTEBOOK_DIR.parent / ".env"
COLLECTION_NAME = os.environ.get("MEM0_PY_COLLECTION", "MEM0_PY_MEMORY_DEMO")
EMBEDDING_DIMS = int(os.environ.get("ORACLE_DB_EMBEDDING_DIMENSION", "32"))

# Keep notebook output clean and avoid telemetry network calls during local runs.
os.environ.setdefault("MEM0_TELEMETRY", "false")
os.environ.setdefault("ANONYMIZED_TELEMETRY", "false")
logging.getLogger("mem0").setLevel(logging.ERROR)
logging.getLogger("posthog").setLevel(logging.CRITICAL)

def show_memory_rows(label, response):
    rows = response.get("results", response) if isinstance(response, dict) else response
    print(f"{label}: {len(rows)} result(s)")
    for idx, row in enumerate(rows, start=1):
        memory = row.get("memory") or row.get("payload", {}).get("data")
        score = row.get("score")
        score_text = f" | score={score:.3f}" if isinstance(score, (int, float)) else ""
        print(f"  {idx}. {memory}{score_text}")


print("Environment file found:", ENV_PATH.exists())
print("Collection:", COLLECTION_NAME)
print("Embedding dimensions:", EMBEDDING_DIMS)

## 2. Confirm Python Dependencies

The notebook uses `mem0ai` for the memory API, `oracledb` for Oracle connectivity, and `python-dotenv` to load local credentials. If a package is missing, this cell installs it into the active Python environment.

The output confirms that the Python tutorial has the required libraries before any database code runs.

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "mem0": "mem0ai",
    "oracledb": "oracledb",
    "dotenv": "python-dotenv",
}

missing = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)

for module in required_packages:
    print(f"{module}: ready")

## 3. Configure the Oracle Connection Profile

This cell loads the private `.env` file and validates the Oracle connection variables. It only prints safe connection metadata, not the username or password.

The result proves that the notebook has everything it needs to connect to Oracle AI Database.

In [ ]:
from dotenv import load_dotenv

load_dotenv(ENV_PATH)

required_env = ["ORACLE_USER", "ORACLE_PASSWORD", "ORACLE_DSN"]
missing_env = [key for key in required_env if not os.environ.get(key)]
if missing_env:
    raise RuntimeError(f"Missing required environment variables: {missing_env}")

ORACLE_USER = os.environ["ORACLE_USER"]
ORACLE_PASSWORD = os.environ["ORACLE_PASSWORD"]
ORACLE_DSN = os.environ["ORACLE_DSN"]
CREATE_INDEX = os.environ.get("MEM0_PY_CREATE_INDEX", "false").lower() in {"1", "true", "yes"}

print("Oracle profile loaded.")
print("DSN:", ORACLE_DSN)
print("Create vector index:", CREATE_INDEX)

## 4. Run a Database Preflight Check

Before using Mem0, the notebook verifies that the schema can create, insert into, select from, and drop a normal table. This keeps local and FreeSQL failures easy to diagnose.

If this cell succeeds, the Oracle credentials and schema privileges are ready for the memory collection.

In [ ]:
import oracledb

oracledb.defaults.program = "devrel-developerhub-mem0-python-oracle-memory"

PREFLIGHT_TABLE = "MEM0_PY_PREFLIGHT"

with oracledb.connect(user=ORACLE_USER, password=ORACLE_PASSWORD, dsn=ORACLE_DSN) as conn:
    with conn.cursor() as cur:
        try:
            cur.execute(f"DROP TABLE {PREFLIGHT_TABLE} PURGE")
        except oracledb.DatabaseError:
            pass
        cur.execute(f"CREATE TABLE {PREFLIGHT_TABLE} (id NUMBER GENERATED BY DEFAULT AS IDENTITY, note VARCHAR2(100))")
        cur.execute(f"INSERT INTO {PREFLIGHT_TABLE} (note) VALUES (:note)", note="preflight ok")
        conn.commit()
        cur.execute(f"SELECT COUNT(*) FROM {PREFLIGHT_TABLE}")
        preflight_count = cur.fetchone()[0]
        cur.execute(f"DROP TABLE {PREFLIGHT_TABLE} PURGE")

print("Database preflight rows:", preflight_count)

## 5. Create a Local Tutorial Embedder

Mem0 normally calls a configured embedding model, then writes the resulting vector to the selected vector store. For this notebook, a small deterministic embedder keeps the tutorial runnable without external model credentials.

The code below registers that local embedder in the Mem0 factory while still using the real Mem0 Python `Memory` API and the real Oracle vector-store provider. In production, replace this with your preferred embedding provider.

In [ ]:
from mem0.embeddings.base import EmbeddingBase
from mem0.utils.factory import EmbedderFactory

class LocalHashEmbedding(EmbeddingBase):
    """Small deterministic embedder for repeatable tutorial runs."""

    def embed(self, text, memory_action=None):
        dims = int(getattr(self.config, "embedding_dims", None) or EMBEDDING_DIMS)
        buckets = [0.0] * dims
        tokens = str(text).lower().replace("-", " ").replace(".", " ").split()

        for token in tokens:
            digest = hashlib.sha256(token.encode("utf-8")).digest()
            index = int.from_bytes(digest[:4], "big") % dims
            sign = 1.0 if digest[4] % 2 == 0 else -1.0
            buckets[index] += sign

        norm = math.sqrt(sum(value * value for value in buckets)) or 1.0
        return [value / norm for value in buckets]

# Mem0 validates known provider names first. For this local tutorial run, reuse
# the allowed fastembed slot and point it at the deterministic embedder above.
EmbedderFactory.provider_to_class["fastembed"] = "__main__.LocalHashEmbedding"

print("Local tutorial embedder registered with Mem0.")

## 6. Configure Mem0 for Oracle AI Database

This cell creates the Mem0 configuration dictionary. The important part is the `vector_store` section: it selects `oracledb`, passes the Oracle connection parameters, sets the collection table name, and tells Oracle the vector dimension.

The notebook also drops the demo collection before recreating it. That makes each run deterministic and keeps the proof-session counts easy to compare.

In [ ]:
def drop_table_if_exists(table_name):
    with oracledb.connect(user=ORACLE_USER, password=ORACLE_PASSWORD, dsn=ORACLE_DSN) as conn:
        with conn.cursor() as cur:
            try:
                cur.execute(f"DROP TABLE {table_name} PURGE")
                print(f"Dropped existing table: {table_name}")
            except oracledb.DatabaseError:
                print(f"No existing table to drop: {table_name}")

# Clean up possible tables from previous runs.
drop_table_if_exists(COLLECTION_NAME)
drop_table_if_exists(f"{COLLECTION_NAME}_ENTITY")

mem0_config = {
    "vector_store": {
        "provider": "oracledb",
        "config": {
            "collection_name": COLLECTION_NAME,
            "embedding_model_dims": EMBEDDING_DIMS,
            "connection_params": {
                "user": ORACLE_USER,
                "password": ORACLE_PASSWORD,
                "dsn": ORACLE_DSN,
            },
            "distance_metric": "COSINE",
            "do_create_index": CREATE_INDEX,
        },
    },
    "embedder": {
        "provider": "fastembed",
        "config": {
            "embedding_dims": EMBEDDING_DIMS,
        },
    },
}

safe_config = json.loads(json.dumps(mem0_config))
safe_config["vector_store"]["config"]["connection_params"] = {
    "user": "***",
    "password": "***",
    "dsn": ORACLE_DSN,
}
pprint(safe_config)

## 7. Initialize Mem0 Memory

`Memory.from_config(...)` creates the Mem0 memory client. At this point, Mem0 creates the Oracle collection table if it does not already exist.

The output confirms that Python is now connected to Mem0 with Oracle AI Database as the backing vector store.

In [ ]:
from mem0 import Memory

memory = Memory.from_config(mem0_config)
print("Mem0 Python Memory initialized.")
print("Vector store:", mem0_config["vector_store"]["provider"])
print("Collection:", COLLECTION_NAME)

## 8. Add User-Scoped Memories

This cell adds explicit memories for two users. `User A` and `User B` each get their own `user_id`, so the same Oracle collection can store memories for multiple users while searches remain scoped.

`infer=False` means Mem0 stores the provided text as the memory. That keeps this notebook focused on the Oracle-backed memory lifecycle rather than LLM-based memory extraction.

In [ ]:
user_a_id = "user_a"
user_b_id = "user_b"

user_a_memories = [
    ("User A prefers vegetarian pasta for weekday dinners.", {"category": "food", "source": "profile"}),
    ("User A likes science fiction movies, especially space exploration stories.", {"category": "movie", "source": "chat"}),
    ("User A is planning a trip to Tokyo in November.", {"category": "travel", "source": "chat"}),
]

user_b_memories = [
    ("User B prefers documentaries about ancient history.", {"category": "movie", "source": "profile"}),
    ("User B avoids seafood when choosing restaurants.", {"category": "food", "source": "chat"}),
]

added_user_a = []
added_user_b = []

for text, metadata in user_a_memories:
    result = memory.add(text, user_id=user_a_id, metadata=metadata, infer=False)
    added_user_a.extend(result["results"])

for text, metadata in user_b_memories:
    result = memory.add(text, user_id=user_b_id, metadata=metadata, infer=False)
    added_user_b.extend(result["results"])

print("User A memories added:", len(added_user_a))
print("User B memories added:", len(added_user_b))
print("Example User A memory id:", added_user_a[0]["id"])

## 9. Search Memories by User Scope

A memory application should retrieve only the memories that belong to the active user or session. The next search uses `filters={"user_id": user_a_id}` so the query only returns `User A` memories.

This result proves that user-scoped retrieval works even though both users are stored in the same Oracle collection.

In [ ]:
user_a_search = memory.search(
    "What should I remember about User A's dinner preferences and movie interests?",
    top_k=3,
    threshold=0.0,
    filters={"user_id": user_a_id},
)

show_memory_rows("User A scoped search", user_a_search)

## 10. Filter Memories by Metadata

Mem0 stores the metadata with each memory payload. Oracle can use that JSON payload to filter results, such as returning only memories where `category` is `movie`.

The output proves that application metadata can narrow memory search beyond the user scope.

In [ ]:
movie_search = memory.search(
    "What movies does this user like?",
    top_k=5,
    threshold=0.0,
    filters={"user_id": user_a_id, "category": "movie"},
)

show_memory_rows("User A movie metadata search", movie_search)

## 11. Update a Memory

Memory systems need lifecycle controls because user preferences change. This cell updates one `User A` memory from a general movie preference to a more specific one.

The follow-up search shows the updated text, which proves the memory payload and vector can be changed in Oracle through the Mem0 Python API.

In [ ]:
movie_memory_id = next(row["id"] for row in added_user_a if row["memory"].startswith("User A likes science fiction"))

update_result = memory.update(
    movie_memory_id,
    text="User A likes optimistic science fiction movies about space exploration.",
    metadata={"category": "movie", "source": "updated_chat"},
)

updated_movie_search = memory.search(
    "What kind of science fiction movies does User A like?",
    top_k=3,
    threshold=0.0,
    filters={"user_id": user_a_id, "category": "movie"},
)

print(update_result["message"])
show_memory_rows("Updated User A movie search", updated_movie_search)

## 12. Delete a Memory

This cell deletes one `User B` memory and then searches again. The before-and-after counts make the lifecycle easy to verify.

The result proves that Mem0 Python can remove a memory from the Oracle collection by memory id.

In [ ]:
user_b_before_delete = memory.search(
    "What should I remember about User B?",
    top_k=5,
    threshold=0.0,
    filters={"user_id": user_b_id},
)

user_b_food_id = next(row["id"] for row in added_user_b if "seafood" in row["memory"])
delete_result = memory.delete(user_b_food_id)

user_b_after_delete = memory.search(
    "What should I remember about User B?",
    top_k=5,
    threshold=0.0,
    filters={"user_id": user_b_id},
)

print(delete_result["message"])
print("User B search results before delete:", len(user_b_before_delete["results"]))
print("User B search results after delete:", len(user_b_after_delete["results"]))
show_memory_rows("Remaining User B memories", user_b_after_delete)

## 13. Build a Memory-Aware Answer

A common application pattern is to retrieve relevant memories first, then pass them into an assistant prompt or answer builder. The next cell keeps that final step local so the notebook remains deterministic.

The answer cites retrieved memories for `User A`, showing how Oracle-backed Mem0 search can personalize an application response.

In [ ]:
question = "Can you suggest a dinner and movie night for User A?"
retrieved = memory.search(
    question,
    top_k=3,
    threshold=0.0,
    filters={"user_id": user_a_id},
)
retrieved_memories = [row["memory"] for row in retrieved["results"]]

answer = (
    "For User A, suggest vegetarian pasta for dinner and an optimistic "
    "space-exploration science fiction movie afterward."
)

print("Question:", question)
print("Retrieved memories:")
for item in retrieved_memories:
    print("-", item)
print("Answer:", answer)

## 14. Inspect the Oracle Collection

The Mem0 Oracle provider stores each memory as a row with an id, vector, and JSON payload. This cell inspects the table and prints safe metadata about the stored rows.

The output proves that the Python memory workflow persisted data inside Oracle AI Database, not only inside notebook variables.

In [ ]:
def inspect_collection(table_name):
    with oracledb.connect(user=ORACLE_USER, password=ORACLE_PASSWORD, dsn=ORACLE_DSN) as conn:
        with conn.cursor() as cur:
            cur.execute(
                """
                SELECT table_name
                FROM user_tables
                WHERE table_name = :table_name
                """,
                table_name=table_name.upper(),
            )
            exists = cur.fetchone() is not None

            if not exists:
                return {"exists": False, "rows": 0, "columns": []}

            cur.execute(f"SELECT COUNT(*) FROM {table_name}")
            row_count = cur.fetchone()[0]

            cur.execute(
                """
                SELECT column_name, data_type
                FROM user_tab_columns
                WHERE table_name = :table_name
                ORDER BY column_id
                """,
                table_name=table_name.upper(),
            )
            columns = cur.fetchall()

            cur.execute(
                f"""
                SELECT JSON_VALUE(payload, '$.user_id'),
                       JSON_VALUE(payload, '$.category'),
                       JSON_VALUE(payload, '$.data')
                FROM {table_name}
                ORDER BY JSON_VALUE(payload, '$.user_id'), JSON_VALUE(payload, '$.category')
                FETCH FIRST 10 ROWS ONLY
                """
            )
            samples = cur.fetchall()

    return {"exists": exists, "rows": row_count, "columns": columns, "samples": samples}

collection_info = inspect_collection(COLLECTION_NAME)
print("Oracle table exists:", collection_info["exists"])
print("Table rows:", collection_info["rows"])
print("Columns:")
for name, data_type in collection_info["columns"]:
    print(f"- {name}: {data_type}")
print("Sample payload data:")
for user_id, category, data in collection_info["samples"]:
    print(f"- {user_id} | {category} | {data}")

## 15. Proof Session

This final check summarizes the key claims from the notebook: scoped search works, metadata filtering works, update/delete works, and Oracle contains the persisted rows.

Use this proof session when writing the blog, because it gives concrete results from the same code path the tutorial explains.

In [ ]:
proof = {
    "user_a_scoped_search_results": len(user_a_search["results"]),
    "user_a_movie_filter_results": len(movie_search["results"]),
    "user_b_before_delete_results": len(user_b_before_delete["results"]),
    "user_b_after_delete_results": len(user_b_after_delete["results"]),
    "oracle_table_exists": collection_info["exists"],
    "oracle_table_rows": collection_info["rows"],
    "lifecycle_update_delete": "PASS" if len(user_b_after_delete["results"]) < len(user_b_before_delete["results"]) else "CHECK",
}

pprint(proof)

## What This Notebook Proves

- A Python application can use `mem0ai` with Oracle AI Database as the vector store.
- Memories can be scoped by `user_id`, so `User A` and `User B` stay separate in retrieval.
- Metadata such as `category` can be stored in the memory payload and used for filtered search.
- Memory lifecycle operations work through the Python API: add, search, update, and delete.
- Oracle stores the memory ids, vectors, and JSON payloads in a database table that can be inspected directly.

## Where This Integration Can Go Next

- Swap the deterministic tutorial embedder for a production embedding provider such as OpenAI, Cohere, OCI Generative AI, Hugging Face, or another provider supported by your application.
- Turn `infer=True` when you want Mem0 to extract memories from conversations with an LLM instead of storing explicit text.
- Add agent-scoped memory by using `agent_id` or `run_id` filters for assistant workflows.
- Create vector indexes for larger collections by setting `MEM0_PY_CREATE_INDEX=true` and tuning the Oracle index parameters.
- Use expiration dates, cleanup jobs, and tenant-aware filters for production memory governance.

## Operational Notes

- Keep database credentials in `.env`, a secret manager, or a managed connection profile. Do not hard-code them in notebooks or source files.
- Use a dedicated collection name for each tutorial, test, or application environment.
- For small local demos, disabling vector index creation can make reruns simpler. For larger collections, enable and tune Oracle vector indexes.
- Use `infer=False` for deterministic tests or explicit memories. Use `infer=True` when your application needs LLM-based memory extraction.
- Keep the same `user_id`, `agent_id`, and metadata strategy in the notebook and blog so readers can map the explanation back to the results.

## References

- [Mem0 Python package](https://pypi.org/project/mem0ai/)
- [Mem0 open-source Python quickstart](https://github.com/mem0ai/mem0/blob/main/docs/open-source/python-quickstart.mdx)
- [Oracle Database Mem0 integration overview](https://docs.oracle.com/en/database/oracle/oracle-database/26/aintg/mem0-oracledb-integration-guide/mem0.html)
- [Mem0 Python with Oracle AI Database](https://docs.oracle.com/en/database/oracle/oracle-database/26/aintg/mem0-oracledb-integration-guide/mem0-python.html)
- [python-oracledb documentation](https://python-oracledb.readthedocs.io/)